# M3 — set-aware conditional low-rank Gaussian copula

This notebook is self-contained. M3 retains M2's covariance family but changes how the full static group is represented before loadings are predicted. See [Chapter 4](../../docs/scientific/04_dependence_models.md).

## Notation and distinction from M2

$i$ indexes origin $t^{(i)}$, $g$ indexes $\mathcal E_g=\{k_1,\ldots,k_{K_g}\}$, and $\tau$ is one-based lead. $Y$ and $y$ denote future random variables and observations, $F$ is the fixed marginal, and $A_{g,\tau}^{(i)}=\sum_kY_{k,\tau}^{(i)}$. Define $V_{k,\tau}^{(i)}\in\{0,1\}$ and $V_{g,\tau}^{(i)}=\prod_kV_{k,\tau}^{(i)}$. When it equals one, $X=[v_{k,\tau}^{(i)}]_k\in\mathbb R^{K_g\times F}$ and $\mathbf z\in\mathbb R^{K_g}$. M2 maps each row independently before forming $R$; M3 first computes $H=\operatorname{Encoder}_\theta(X)$ so each representation may depend on the full group. A shared head predicts $(\lambda_k,b_k)$ and $\Sigma=\Lambda\Lambda^\mathsf T+\operatorname{diag}(\sigma^2)$.

## Self-attention and permutation equivariance

For one attention head, let $Q^{att}=XW_Q$, $K^{att}=XW_K$, and $B^{att}=XW_V$. Then
$$\operatorname{Attn}(X)=\operatorname{softmax}(Q^{att}(K^{att})^{\mathsf T}/\sqrt d)B^{att}.$$
Each row is a weighted combination of all entity value vectors. No positional encoding or entity-ID embedding is used. For a permutation matrix $P$, the desired identity is $R(PX)=PR(X)P^\mathsf T$. This means reordering entity records changes only matrix order, not the physical model.

In [ ]:
from pathlib import Path
import sys, numpy as np, matplotlib.pyplot as plt
PROJECT_ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(PROJECT_ROOT/'src') not in sys.path: sys.path.insert(0,str(PROJECT_ROOT/'src'))
from simcast.config import load_base_config, load_method_config, resolve_run_config
from simcast.experiments import locate_compatible_cache
from simcast.fm.cache import load_pit_library
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config
BASE_FILE='configs/bases/liander2024/transformer.yaml'
METHOD_FILE='configs/methods/m3_set_aware_low_rank.yaml'
BASE_OVERRIDES=(); METHOD_OVERRIDES=()
TRAIN_IF_MISSING=False; RUN_EVALUATION=False
RUN_DIR=PROJECT_ROOT/'runs'/'notebook_walkthrough'/'m3_set_aware'
base=load_base_config(PROJECT_ROOT/BASE_FILE,overrides=BASE_OVERRIDES)
method=load_method_config(PROJECT_ROOT/METHOD_FILE,overrides=METHOD_OVERRIDES)
config=resolve_run_config(base,method)
cache_dir=locate_compatible_cache(base)
library=load_pit_library(cache_dir,access='training'); ds=library.dataset
entity_ids=[str(x) for x in ds.entity_id.values]; K_g=len(entity_ids)
s=method.model
print({'g':base.data.entity_type,'K_g':K_g,'layers':s.num_layers,'heads':s.num_heads,'rank':s.latent_rank})


## Attention as contextual weighting

The following synthetic matrix is an explanatory example, not a learned result. Row $a$, column $b$ is the fraction of entity $a$'s contextual update drawn from entity $b$ in one head. Rows sum to one. Actual multi-head, multi-layer attention is not itself a correlation matrix; the final loadings determine $R$.

In [ ]:
weights=np.array([[.55,.35,.10],[.30,.55,.15],[.10,.20,.70]])
fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(weights,vmin=0,vmax=1,cmap='viridis')
for a in range(3):
    for b in range(3): ax.text(b,a,f'{weights[a,b]:.2f}',ha='center',va='center',color='white')
ax.set(title='Illustrative attention weights',xlabel='entity supplying information',ylabel='entity being contextualized'); fig.colorbar(im,ax=ax); plt.show()


## Estimation and optional computation

M3 uses the same finite pseudo-likelihood and chronological checkpoint selection as M2. Its additional hypothesis is that cross-entity context contains predictive information about current dependence. The cell below is inert unless explicitly enabled and delegates to the same authoritative functions as the CLI.

In [ ]:
if TRAIN_IF_MISSING and not (RUN_DIR/'best.pt').is_file():
    train_from_config(config,cache_dir=cache_dir,output_dir=RUN_DIR)
if RUN_EVALUATION:
    evaluate_from_config(config,methods=('set_aware_low_rank',),method_runs={'set_aware_low_rank':RUN_DIR},cache_dir=cache_dir,output_dir=RUN_DIR/'evaluation')


## Controls and interpretation

`method.model.model_dim`, `num_layers`, `num_heads`, `latent_rank`, `dropout`, `sigma_floor`, and `jitter` define the model. `method.features.*` defines $X$ and `method.optimization.*` defines estimation. `SetAwareLowRankGaussianCopula` implements M3. Its permutation test is a mathematical invariance test, not a reduced-cardinality experiment. A gain over M2 isolates the value of full-group contextualization within the shared low-rank family.